In [2]:
!apt-get update
!apt-get install -y unrar ffmpeg

!pip install -q torch torchvision torchaudio
!pip install -q pytorchvideo
!pip install -q rarfile
!pip install -q tqdm
!pip install -q scikit-learn
!pip install -q pandas matplotlib seaborn
!pip install -q opencv-python-headless
!pip install -q pillow

import requests
import os
import rarfile

# ссылки на файлы
anomaly_url = "https://disk.yandex.ru/d/n2xs9DCwgFE36Q"
normal_url = "https://disk.yandex.ru/d/ySzyhHhF2SsYgA"

def get_direct_link(public_url):
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
    params = {"public_key": public_url}
    try:
        response = requests.get(api_url, params=params)
        response.raise_for_status()
        return response.json()["href"]
    except Exception as e:
        print(f"Ошибка: {e}")
        return None

print("Получение ссылок для скачивания...")
anomaly_direct = get_direct_link(anomaly_url)
normal_direct = get_direct_link(normal_url)

if not anomaly_direct or not normal_direct:
    print("Не удалось получить ссылки. Проверь ссылки на Яндекс.Диске.")
else:
    # скачиваем
    print("\nСкачивание anomaly.rar (5.3 GB)...")
    !wget -O /content/anomaly.rar "{anomaly_direct}" --show-progress

    print("\nСкачивание normal.rar (12.3 GB)...")
    !wget -O /content/normal.rar "{normal_direct}" --show-progress

    # проверка размера
    print("\nРазмеры файлов:")
    for name in ['anomaly.rar', 'normal.rar']:
        path = f'/content/{name}'
        if os.path.exists(path):
            size = os.path.getsize(path) / (1024**3)
            print(f"{name}: {size:.2f} GB")

    # распаковка
    print("\nРаспаковка anomaly.rar...")
    with rarfile.RarFile('/content/anomaly.rar') as rf:
        rf.extractall('/content/Anomaly_Videos')

    print("Распаковка normal.rar...")
    with rarfile.RarFile('/content/normal.rar') as rf:
        rf.extractall('/content/Normal_Videos')

    # проверка
    print("\nПроверка структуры:")
    anomaly_dirs = os.listdir('/content/Anomaly_Videos')
    normal_count = len(os.listdir('/content/Normal_Videos'))
    print(f"Anomaly_Videos: {len(anomaly_dirs)} папок (категорий)")
    print(f"Normal_Videos: {normal_count} видео")

    # подсчёт видео в аномалиях
    total_anomaly = 0
    for cat in anomaly_dirs[:3]:
        cat_path = os.path.join('/content/Anomaly_Videos', cat)
        if os.path.isdir(cat_path):
            count = len(os.listdir(cat_path))
            print(f"  {cat}: {count} видео")
            total_anomaly += count

    print(f"\nДанные готовы к обучению!")
    print(f"Всего аномальных видео: {total_anomaly}+")
    print(f"Всего нормальных видео: {normal_count}")


Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,083 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,308 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted a

In [3]:
!git clone https://github.com/turbinn/anomaly-detection-ucf-crime.git
%cd anomaly-detection-ucf-crime/project

Cloning into 'anomaly-detection-ucf-crime'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 58 (delta 16), reused 51 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 32.52 KiB | 16.26 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/anomaly-detection-ucf-crime/project


In [5]:
%%writefile /content/anomaly-detection-ucf-crime/project/src/__init__.py
from .data_loader import VideoDataset, load_ucf_crime_data, create_dataloaders
from .models import get_model, get_model_params, get_recommended_batch_size, CNNLSTM
from .train import train_model
from .utils import AverageMeter, compute_metrics, save_metrics, save_checkpoint, load_checkpoint

Overwriting /content/anomaly-detection-ucf-crime/project/src/__init__.py


In [11]:
%cd /content/anomaly-detection-ucf-crime/project

!git pull

/content/anomaly-detection-ucf-crime/project
Already up to date.


In [22]:
%cd /content/anomaly-detection-ucf-crime/project
!git checkout -- src/models.py
!git pull

/content/anomaly-detection-ucf-crime/project
Already up to date.


In [19]:
%cd /content/anomaly-detection-ucf-crime/project
!python scripts/train_all_models.py

/content/anomaly-detection-ucf-crime/project

Training r3d_18
Training r3d_18 with batch_size=8
Anomaly path: /content/Anomaly_Videos
Normal path: /content/Normal_Videos
Found 201 videos (0 anomaly, 201 normal)
Train samples: 160, Test samples: 41
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Total params: 33,167,298, Trainable params: 33,167,298

Epoch 1/3
Training: 100% 20/20 [00:33<00:00,  1.69s/it, Loss=0.262, Acc=98.7

In [44]:
%cd /content/anomaly-detection-ucf-crime/project
!git pull

/content/anomaly-detection-ucf-crime/project
Already up to date.


In [46]:
%%writefile /content/anomaly-detection-ucf-crime/project/src/data_loader.py
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import cv2
import numpy as np

class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, model_name="r3d_18", frames_per_video=16, is_train=True):
        self.video_paths = video_paths
        self.labels = labels
        self.model_name = model_name
        self.frames_per_video = frames_per_video
        self.is_train = is_train

        self.resize_size = self._get_resize_size()
        self.transform = self._get_transforms()

    def _get_resize_size(self):
        sizes = {
            "r3d_18": 112,
            "mc3_18": 112,
            "r2plus1d_18": 112,
            "mvit": 224,
            "s3d": 112,
            "cnn_lstm": 112
        }
        return sizes.get(self.model_name, 112)

    def _get_transforms(self):
        if self.is_train:
            return transforms.Compose([
                transforms.Resize((self.resize_size, self.resize_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.43216, 0.394666, 0.37645],
                                   std=[0.22803, 0.22145, 0.216989])
            ])
        else:
            return transforms.Compose([
                transforms.Resize((self.resize_size, self.resize_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.43216, 0.394666, 0.37645],
                                   std=[0.22803, 0.22145, 0.216989])
            ])

    def _extract_frames(self, video_path):
        frames = []
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            return None

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
            cap.release()
            return None

        indices = np.linspace(0, total_frames - 1, self.frames_per_video, dtype=int)

        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = Image.fromarray(frame)
                frame = self.transform(frame)
                frames.append(frame)
            else:
                frames.append(torch.zeros(3, self.resize_size, self.resize_size))

        cap.release()

        if len(frames) == 0:
            return None

        return torch.stack(frames)

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        frames = self._extract_frames(video_path)

        if frames is None:
            frames = torch.zeros(self.frames_per_video, 3, self.resize_size, self.resize_size)

        # для 3D CNN и MViT: [frames, channels, H, W] -> [channels, frames, H, W]
        if self.model_name in ["r3d_18", "mc3_18", "r2plus1d_18", "s3d", "mvit"]:
            frames = frames.permute(1, 0, 2, 3)

        return frames, label


def load_ucf_crime_data(data_root, model_name="r3d_18", frames_per_video=16):
    anomaly_dir = os.path.join(data_root, "Anomaly_Videos")
    normal_dir = os.path.join(data_root, "Normal_Videos")

    if not os.path.exists(anomaly_dir):
        for item in os.listdir(data_root):
            if 'anomaly' in item.lower():
                anomaly_dir = os.path.join(data_root, item)
                break

    if not os.path.exists(normal_dir):
        for item in os.listdir(data_root):
            if 'normal' in item.lower():
                normal_dir = os.path.join(data_root, item)
                break

    print(f"Anomaly path: {anomaly_dir}")
    print(f"Normal path: {normal_dir}")

    video_paths = []
    labels = []

    if os.path.exists(anomaly_dir):
        items = os.listdir(anomaly_dir)
        has_subdirs = any(os.path.isdir(os.path.join(anomaly_dir, item)) for item in items)

        if has_subdirs:
            for category in items:
                category_path = os.path.join(anomaly_dir, category)
                if os.path.isdir(category_path):
                    for video_file in os.listdir(category_path):
                        if video_file.endswith(('.mp4', '.avi', '.mkv')):
                            video_paths.append(os.path.join(category_path, video_file))
                            labels.append(1)
        else:
            for video_file in items:
                if video_file.endswith(('.mp4', '.avi', '.mkv')):
                    video_paths.append(os.path.join(anomaly_dir, video_file))
                    labels.append(1)

    if os.path.exists(normal_dir):
        for video_file in os.listdir(normal_dir):
            if video_file.endswith(('.mp4', '.avi', '.mkv')):
                video_paths.append(os.path.join(normal_dir, video_file))
                labels.append(0)

    print(f"Found {len(video_paths)} videos ({sum(labels)} anomaly, {len(labels)-sum(labels)} normal)")

    return video_paths, labels


def create_dataloaders(data_root, model_name="r3d_18", batch_size=8, frames_per_video=16,
                       train_split=0.8, num_workers=2):
    video_paths, labels = load_ucf_crime_data(data_root, model_name, frames_per_video)

    indices = np.random.permutation(len(video_paths))
    split_idx = int(len(video_paths) * train_split)

    train_indices = indices[:split_idx]
    test_indices = indices[split_idx:]

    train_paths = [video_paths[i] for i in train_indices]
    train_labels = [labels[i] for i in train_indices]
    test_paths = [video_paths[i] for i in test_indices]
    test_labels = [labels[i] for i in test_indices]

    train_dataset = VideoDataset(
        train_paths, train_labels, model_name, frames_per_video, is_train=True
    )
    test_dataset = VideoDataset(
        test_paths, test_labels, model_name, frames_per_video, is_train=False
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )

    return train_loader, test_loader, len(train_dataset), len(test_dataset)

Overwriting /content/anomaly-detection-ucf-crime/project/src/data_loader.py


In [47]:
import sys
if 'src.data_loader' in sys.modules:
    del sys.modules['src.data_loader']
if 'src' in sys.modules:
    del sys.modules['src']

In [50]:
%%writefile /content/train_mvit_only.py
import sys
sys.path.append('/content/anomaly-detection-ucf-crime/project')

from src.train import train_model

model, history, best_acc = train_model(
    model_name='mvit',
    data_root='/content',
    epochs=3,
    lr=0.0001,
    batch_size=12,
    frames_per_video=16,
    save_dir='checkpoints'
)

print(f'\nMViT best accuracy: {best_acc:.2f}%')

Writing /content/train_mvit_only.py


In [51]:
%cd /content/anomaly-detection-ucf-crime/project
!python /content/train_mvit_only.py

/content/anomaly-detection-ucf-crime/project
Training mvit with batch_size=12
Anomaly path: /content/Anomaly_Videos
Normal path: /content/Normal_Videos
Found 201 videos (0 anomaly, 201 normal)
Train samples: 160, Test samples: 41
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MViT_V2_S_Weights.KINETICS400_V1`. You can also use `weights=MViT_V2_S_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Total params: 34,231,682, Trainable params: 34,231,682

Epoch 1/3
Training: 100% 14/14 [00:37<00:00,  2.69s/it, Loss=0.261, Acc=94.38%]
Validati